**1. REad files using Auto Loader **

In [0]:
customer_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",'json')
    .option("cloudFiles.schemaLocation", "/Volumes/gizmobox/landing/operational_data/customers_autoloader/_schema")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaHints", "date_of_birth DATE, member_since DATE, created_timestamp TIMESTAMP")
    .option("pathGlobFilter", "customers_2024_*.json")
    .load("/Volumes/gizmobox/landing/operational_data/customers_autoloader/")
)

**2. Transform the dataframe to add the following columns**
- file_path : Clound file path
- ingestion_date :  Current timestamp

In [0]:
from pyspark.sql.functions import current_timestamp, col
df_transformed_data = (
    customer_df
    .withColumn("file_path", col("_metadata.file_path"))
    .withColumn("ingestion_date", current_timestamp())
)



**3. Write the transformed data stream to delta table**

In [0]:
streaming_query = (
df_transformed_data.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/gizmobox/landing/operational_data/customers_autoloader/_checkpoint_stream")
    .toTable("gizmobox.bronze.customer_autoloader")
)

In [0]:
df = spark.table("gizmobox.bronze.customer_autoloader")
display(df)

In [0]:
streaming_query.stop()